<a href="https://colab.research.google.com/github/SilkSherstka/hse_text_analysis_and_visualization/blob/main/Andronova_homework_interactive_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание 2
## Интерактивная или нарративная визуализация текстовых данных

Эта тетрадь — типа шаблон для выполнения домашнего задания по теме интерактивной и нарративной визуализации.

## Что нужно сделать
На основе текстового датасета разработать **интерактивную визуализацию** или **визуальную историю**, которая помогает исследовать данные и делает выводы понятными для аудитории.

## Инструменты на выбор
- **Plotly**
- **PyVis**
- **Streamlit**

## Два маршрута выполнения
### Вариант A. Интерактивная визуализация
Например:
- интерактивный граф тематического сходства текстов;
- интерактивная диаграмма частот;
- мини-дашборд с фильтрами и аннотациями.

### Вариант B. Нарративная визуализация
Например:
- пошаговая визуальная история о структуре корпуса;
- последовательный разбор нескольких наблюдений;
- объяснение одного исследовательского вопроса через цепочку графиков.

---

## Что будет оцениваться
- интерактивные / нарративные элементы — **3 балла**
- логика визуального рассказа — **3 балла**
- соответствие визуализации аналитической цели — **2 балла**
- качество реализации и оформления — **2 балла**

**Итого: 10 баллов**


## Как работать с тетрадью

1. Сначала сформулируйте **аналитическую цель**.
2. Затем выберите **маршрут**:
   - интерактивный;
   - нарративный.
3. Подготовьте данные.
4. Постройте основную визуализацию.
5. Добавьте пояснения, аннотации, подписи.
6. В конце напишите короткое объяснение своего решения.

---

## Важно
Эта тетрадь — **рабочий шаблон**, а не готовое решение.  
В ней есть:
- подсказки;
- каркас проекта;
- шаблонные блоки кода;
- чек-лист перед сдачей.


# Блок 1. Постановка задачи

Сначала сформулируйте, **что именно вы хотите показать**.

### Мини-шаблон
Заполните 3 пункта:
1. **Что за данные вы анализируете?**
2. **Какой вопрос задаете данным?**
3. **Почему для этого подходит интерактивный или нарративный формат?**


In [1]:
# Заполните ответы в виде строк

dataset_description = (
    "Корпус из 200 сообщений Usenet (sklearn 20 Newsgroups): 5 тематических "
    "рубрик, для каждого текста посчитаны длина (число слов) и лексическое "
    "разнообразие (доля уникальных слов)."
)
research_question = (
    "Как соотносятся длина текста и лексическое разнообразие в разных темах, "
    "и есть ли темы с заметно более «разнообразным» словарём?"
)
chosen_format = "interactive dashboard (Plotly)"  # маршрут A

print("Описание датасета:", dataset_description)
print("Исследовательский вопрос:", research_question)
print("Формат:", chosen_format)


Описание датасета: Корпус из 200 сообщений Usenet (sklearn 20 Newsgroups): 5 тематических рубрик, для каждого текста посчитаны длина (число слов) и лексическое разнообразие (доля уникальных слов).
Исследовательский вопрос: Как соотносятся длина текста и лексическое разнообразие в разных темах, и есть ли темы с заметно более «разнообразным» словарём?
Формат: interactive dashboard (Plotly)


# Блок 2. Загрузка библиотек

Ниже — базовые импорты.  
Вы можете оставить только те, которые нужны именно вам.

### Подсказка
- **Plotly** удобен для интерактивных графиков.
- **PyVis** удобен для сетей и графов.
- **Streamlit** нужен, если вы делаете мини-приложение.


In [3]:
import pandas as pd
import matplotlib.pyplot as plt

# Plotly
import plotly.express as px
import plotly.graph_objects as go

# PyVis
# Если нужно, раскомментируйте:
# !pip install pyvis
# from pyvis.network import Network

# Общие настройки
pd.set_option("display.max_columns", 50)
%matplotlib inline


# Блок 3. Ваши данные

Загрузите свой датасет.

### Подсказка
Если у вас пока нет готового датасета, можно использовать:
- таблицу с текстами и метаданными;
- таблицу с частотами слов;
- таблицу с темами;
- таблицу со сходством между документами.


In [4]:
from pathlib import Path
import re

CORPUS_CSV = Path("text_corpus_features.csv")

if CORPUS_CSV.exists():
    df = pd.read_csv(CORPUS_CSV)
else:
    from sklearn.datasets import fetch_20newsgroups

    categories = [
        "sci.space",
        "rec.sport.hockey",
        "talk.politics.misc",
        "comp.graphics",
        "alt.atheism",
    ]
    raw = fetch_20newsgroups(
        subset="train",
        categories=categories,
        shuffle=True,
        random_state=42,
        remove=("headers", "footers", "quotes"),
    )
    records = []
    for i, (text, cat_idx) in enumerate(zip(raw.data[:200], raw.target)):
        words = re.findall(r"[A-Za-z']+", text.lower())
        n_words = len(words)
        records.append(
            {
                "doc_id": i + 1,
                "title": (text.strip().split("\n")[0][:60] or f"Doc {i + 1}"),
                "theme": categories[cat_idx],
                "length": n_words,
                "lexical_diversity": round(len(set(words)) / n_words, 3) if n_words else 0,
                "year": 2020 + (i % 4),  # псевдо-период для фильтра (индекс в выборке)
            }
        )
    df = pd.DataFrame(records)
    df.to_csv(CORPUS_CSV, index=False, encoding="utf-8")

df.head()


,doc_id,title,theme,length,lexical_diversity,year
0,1,"G'day all,",rec.sport.hockey,157,0.745,2020
1,2,Are there any further stories to report on the...,talk.politics.misc,111,0.685,2021
2,3,And you know why this is? Because you've conve...,sci.space,43,0.744,2022
3,4,Replying to A.J. Teel:,alt.atheism,852,0.454,2023
4,5,"True. At first, the news media seemed entranc...",sci.space,1345,0.385,2020


# Блок 4. Быстрый обзор данных

Перед визуализацией полезно быстро посмотреть на структуру таблицы.

### Подсказка
Проверьте:
- размер таблицы;
- названия колонок;
- типы данных;
- есть ли пропуски.


In [5]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nInfo:")
display(df.info())
print("\nMissing values:")
display(df.isna().sum())
display(df.head())


Shape: (200, 6)

Columns: ['doc_id', 'title', 'theme', 'length', 'lexical_diversity', 'year']

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   doc_id             200 non-null    int64  
 1   title              200 non-null    object 
 2   theme              200 non-null    object 
 3   length             200 non-null    int64  
 4   lexical_diversity  200 non-null    float64
 5   year               200 non-null    int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 9.5+ KB


None


Missing values:


,0
doc_id,0
title,0
theme,0
length,0
lexical_diversity,0
year,0


,doc_id,title,theme,length,lexical_diversity,year
0,1,"G'day all,",rec.sport.hockey,157,0.745,2020
1,2,Are there any further stories to report on the...,talk.politics.misc,111,0.685,2021
2,3,And you know why this is? Because you've conve...,sci.space,43,0.744,2022
3,4,Replying to A.J. Teel:,alt.atheism,852,0.454,2023
4,5,"True. At first, the news media seemed entranc...",sci.space,1345,0.385,2020


# Блок 5. Выбор маршрута

Ниже два рабочих сценария.  
Вы можете использовать **один** из них или адаптировать оба под свою задачу.

---

## Маршрут A. Интерактивная визуализация
Подходит, если вы хотите, чтобы пользователь:
- сравнивал группы;
- наводил курсор и читал подсказки;
- фильтровал данные;
- переключал режимы просмотра.

## Маршрут B. Нарративная визуализация
Подходит, если вы хотите:
- провести аудиторию шаг за шагом;
- показать цепочку наблюдений;
- объяснить структуру корпуса;
- построить визуальную историю с выводом.


# Маршрут A. Интерактивная визуализация

## Задание A1
Постройте базовую интерактивную визуализацию в Plotly.

### Подсказка
Выберите один формат:
- scatter plot;
- bar chart;
- line chart;
- box plot.

Подумайте, какие переменные вы хотите сравнить.


In [6]:
fig = px.scatter(
    df,
    x="length",
    y="lexical_diversity",
    color="theme",
    hover_data={"title": True, "year": True, "doc_id": True, "length": ":,", "lexical_diversity": ":.3f"},
    labels={
        "length": "Длина текста (слова)",
        "lexical_diversity": "Лексическое разнообразие",
        "theme": "Тема",
    },
    title="Длина текста и лексическое разнообразие по темам (20 Newsgroups)",
    template="plotly_white",
    height=520,
)
fig.update_traces(marker=dict(size=10, opacity=0.85, line=dict(width=0.5, color="white")))
fig.update_layout(legend_title_text="Тема")
fig.show()


## Задание A2
Добавьте элемент управления или дополнительную интерактивность.

### Что можно добавить
- фильтрацию по году;
- отдельные подписи;
- hover с пояснением;
- выбор категории через dropdown;
- faceting по теме.


In [7]:
THEME_LABELS = {
    "sci.space": "Космос",
    "rec.sport.hockey": "Хоккей",
    "talk.politics.misc": "Политика",
    "comp.graphics": "Графика",
    "alt.atheism": "Атеизм",
}
plot_df = df.assign(theme_ru=df["theme"].map(THEME_LABELS))

years = sorted(plot_df["year"].unique())
fig2 = go.Figure()
trace_meta = []
for year in years:
    for theme in sorted(plot_df["theme_ru"].unique()):
        part = plot_df[(plot_df["year"] == year) & (plot_df["theme_ru"] == theme)]
        if part.empty:
            continue
        fig2.add_trace(
            go.Scatter(
                x=part["length"],
                y=part["lexical_diversity"],
                mode="markers",
                name=theme,
                marker=dict(size=10, opacity=0.85),
                customdata=part[["title", "doc_id", "theme"]].values,
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Тема: %{customdata[2]}<br>"
                    "Длина: %{x} слов<br>"
                    "Разнообразие: %{y:.3f}<br>"
                    "ID: %{customdata[1]}<extra></extra>"
                ),
                visible=(year == years[0]),
            )
        )
        trace_meta.append(year)

buttons = []
for y in years:
    visible = [meta == y for meta in trace_meta]
    buttons.append(
        dict(
            label=str(y),
            method="update",
            args=[{"visible": visible}, {"title": f"Корпус за {y} год (псевдо-период в выборке)"}],
        )
    )

# средние по темам (всегда видны)
means = plot_df.groupby("theme_ru", as_index=False).agg(
    length=("length", "mean"), lexical_diversity=("lexical_diversity", "mean")
)
fig2.add_trace(
    go.Scatter(
        x=means["length"],
        y=means["lexical_diversity"],
        mode="markers+text",
        name="Среднее по теме",
        marker=dict(symbol="diamond", size=14, color="black", line=dict(width=2, color="white")),
        text=means["theme_ru"],
        textposition="top center",
        hovertemplate="Среднее: %{text}<br>Длина: %{x:.0f}<br>Разнообразие: %{y:.3f}<extra></extra>",
    )
)
for btn in buttons:
    btn["args"][0]["visible"].append(True)  # ромбы со средними всегда на графике

fig2.update_layout(
    title=f"Корпус за {years[0]} год (псевдо-период в выборке)",
    xaxis_title="Длина текста (слова)",
    yaxis_title="Лексическое разнообразие",
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            x=0.02,
            y=1.12,
            showactive=True,
            xanchor="left",
            yanchor="top",
        )
    ],
    annotations=[
        dict(
            text="Период:",
            x=0.02,
            y=1.18,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(size=12),
        ),
        dict(
            text="comp.graphics — самые длинные тексты; rec.sport.hockey — короче, но разнообразнее",
            x=0.5,
            y=-0.12,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(size=11, color="#444"),
        ),
    ],
    template="plotly_white",
    height=560,
    legend_title_text="Тема",
)

fig2.show()


## Задание A3
Напишите 3–5 предложений:
- что показывает график;
- что пользователь может исследовать сам;
- почему выбранный формат помогает анализу.


In [8]:
interactive_explanation = """
График показывает связь между длиной сообщения и лексическим разнообразием: каждая точка —
отдельный документ, цвет кодирует тематическую рубрику Usenet. Видно, что comp.graphics
смещён вправо (длиннее тексты), а rec.sport.hockey — влево и чуть выше по разнообразию.

Пользователь может навести курсор на точку и прочитать заголовок, тему и ID документа,
переключить псевдо-период (год) через выпадающий список и сравнить подвыборки без пересчёта
данных. Чёрные ромбы — средние по теме для всего корпуса.

Интерактивный формат уместен, потому что при 200 точках и 5 темах статичный рисунок
перегружен: hover и фильтр по периоду позволяют исследовать кластеры и выбросы точечно,
не теряя контекст по группам.
"""
print(interactive_explanation)



График показывает связь между длиной сообщения и лексическим разнообразием: каждая точка —
отдельный документ, цвет кодирует тематическую рубрику Usenet. Видно, что comp.graphics
смещён вправо (длиннее тексты), а rec.sport.hockey — влево и чуть выше по разнообразию.

Пользователь может навести курсор на точку и прочитать заголовок, тему и ID документа,
переключить псевдо-период (год) через выпадающий список и сравнить подвыборки без пересчёта
данных. Чёрные ромбы — средние по теме для всего корпуса.

Интерактивный формат уместен, потому что при 200 точках и 5 темах статичный рисунок
перегружен: hover и фильтр по периоду позволяют исследовать кластеры и выбросы точечно,
не теряя контекст по группам.



# Маршрут B. Нарративная визуализация

## Задание B1
Сформулируйте структуру визуального рассказа.

### Подсказка
Удобный шаблон:
1. Вводный график: общий обзор данных
2. Второй график: уточнение или сравнение
3. Третий график: главный вывод

Запишите кратко, что будет происходить на каждом шаге.


In [2]:
# Маршрут B не выбран — работа выполнена по маршруту A (интерактивный Plotly).


## Задание B2
Постройте первый график — обзорный.

### Подсказка
Первый график должен отвечать на вопрос:
**«Что это вообще за данные?»**


In [ ]:
# Ваш код для первого графика



## Задание B3
Постройте второй график — уточняющий.

### Подсказка
На этом шаге вы уже можете:
- сравнивать группы;
- выделять аномалии;
- добавлять подписи;
- переходить от общего к частному.


In [ ]:
# Ваш код для второго графика



## Задание B4
Постройте третий график — финальный вывод.

### Подсказка
Последний график должен подводить к ответу на исследовательский вопрос.


In [ ]:
# Ваш код для третьего графика



## Задание B5
Напишите короткий текст сопровождения:
- что показывает каждый шаг;
- как один график переходит в другой;
- к какому выводу вы подводите зрителя.


In [ ]:
narrative_explanation = '''
'''
print(narrative_explanation)


# Блок 6. Дополнительный маршрут для PyVis

Используйте этот блок, если вы хотите делать **сетевую визуализацию**.

### Когда это уместно
- вы показываете связи между документами;
- вы визуализируете тематическое сходство;
- вы хотите построить граф отношений между сущностями.

### Подсказка
Для начала достаточно:
- списка узлов;
- списка ребер;
- подписи и веса связи.


In [ ]:
# Пример каркаса для PyVis
# !pip install pyvis
# from pyvis.network import Network

# nodes = [
#     (1, "Doc A"),
#     (2, "Doc B"),
#     (3, "Doc C")
# ]

# edges = [
#     (1, 2, 0.81),
#     (1, 3, 0.45),
#     (2, 3, 0.62)
# ]

# net = Network(height="600px", width="100%", notebook=True)
#
# for node_id, label in nodes:
#     net.add_node(node_id, label=label)
#
# for source, target, weight in edges:
#     net.add_edge(source, target, value=weight, title=f"Similarity: {weight}")
#
# net.show("network.html")


# Блок 7. Если вы делаете Streamlit

Если вы решите делать **мини-дашборд**, удобно сначала продумать его структуру.

### Мини-шаблон структуры
- заголовок приложения;
- краткое описание задачи;
- фильтр или selectbox;
- основной график;
- дополнительный график или таблица;
- короткий вывод.

Ниже — каркас файла `app.py`.


In [ ]:
streamlit_template = r'''
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Text Data Dashboard", layout="wide")

st.title("Text Data Dashboard")
st.write("Краткое описание задачи и данных")

# df = pd.read_csv("your_file.csv")

# Учебный пример
df = pd.DataFrame({
    "doc_id": [1, 2, 3, 4, 5],
    "title": ["Doc A", "Doc B", "Doc C", "Doc D", "Doc E"],
    "theme": ["education", "education", "media", "media", "science"],
    "length": [120, 95, 140, 110, 160],
    "lexical_diversity": [0.72, 0.68, 0.61, 0.66, 0.75],
    "year": [2021, 2022, 2022, 2023, 2023]
})

theme_filter = st.selectbox("Choose theme", ["All"] + sorted(df["theme"].unique().tolist()))

if theme_filter != "All":
    filtered_df = df[df["theme"] == theme_filter]
else:
    filtered_df = df.copy()

fig = px.scatter(
    filtered_df,
    x="length",
    y="lexical_diversity",
    color="theme",
    hover_data=["title", "year"],
    title="Text length vs lexical diversity"
)

st.plotly_chart(fig, use_container_width=True)
st.dataframe(filtered_df)
'''
print(streamlit_template)



import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Text Data Dashboard", layout="wide")

st.title("Text Data Dashboard")
st.write("Краткое описание задачи и данных")

# df = pd.read_csv("your_file.csv")

# Учебный пример
df = pd.DataFrame({
    "doc_id": [1, 2, 3, 4, 5],
    "title": ["Doc A", "Doc B", "Doc C", "Doc D", "Doc E"],
    "theme": ["education", "education", "media", "media", "science"],
    "length": [120, 95, 140, 110, 160],
    "lexical_diversity": [0.72, 0.68, 0.61, 0.66, 0.75],
    "year": [2021, 2022, 2022, 2023, 2023]
})

theme_filter = st.selectbox("Choose theme", ["All"] + sorted(df["theme"].unique().tolist()))

if theme_filter != "All":
    filtered_df = df[df["theme"] == theme_filter]
else:
    filtered_df = df.copy()

fig = px.scatter(
    filtered_df,
    x="length",
    y="lexical_diversity",
    color="theme",
    hover_data=["title", "year"],
    title="Text length vs lexical diversity"
)

st.plotly_chart(f

# Блок 8. Финальное пояснение к работе

Здесь нужно коротко и внятно объяснить свою работу.

### Напишите:
1. Какую задачу вы решали?
2. Почему выбрали именно этот формат?
3. Что пользователь / зритель должен понять из визуализации?
4. Какой главный вывод можно сделать?


In [ ]:
final_explanation = """
1. Задача: сравнить длину и лексическое разнообразие текстов в пяти темах Usenet и понять,
   отличаются ли рубрики по «объёму» и «богатству» словаря.
2. Формат: интерактивный scatter в Plotly — позволяет фильтровать подвыборки и читать
   метаданные каждого документа без отдельной таблицы.
3. Зритель видит, что comp.graphics — самые длинные сообщения, а спортивная рубрика короче,
   но с более высоким разнообразием; политика и атеизм ближе к центру облака.
4. Главный вывод: тематика связана с профилем текста (длина и лексика), поэтому при
   сравнении корпусов важно контролировать тему, а не смотреть только на общие средние.
"""
print(final_explanation)



1. Задача: сравнить длину и лексическое разнообразие текстов в пяти темах Usenet и понять,
   отличаются ли рубрики по «объёму» и «богатству» словаря.
2. Формат: интерактивный scatter в Plotly — позволяет фильтровать подвыборки и читать
   метаданные каждого документа без отдельной таблицы.
3. Зритель видит, что comp.graphics — самые длинные сообщения, а спортивная рубрика короче,
   но с более высоким разнообразием; политика и атеизм ближе к центру облака.
4. Главный вывод: тематика связана с профилем текста (длина и лексика), поэтому при
   сравнении корпусов важно контролировать тему, а не смотреть только на общие средние.



# Блок 9. Чек-лист перед сдачей

Проверьте, что у вас:
- есть понятная **аналитическая цель**;
- выбран и реализован **один формат**: интерактивный или нарративный;
- визуализация не просто красивая, а действительно отвечает на вопрос;
- есть подписи, заголовки и пояснения;
- есть короткий текст с объяснением решения;
- структура работы читается без вашего устного комментария.


# Типичные ошибки

- интерактивность ради интерактивности;
- слишком много элементов на одном экране;
- отсутствие исследовательского вопроса;
- набор графиков без общей логики;
- слабые подписи и отсутствие пояснений;
- визуализация не соответствует задаче анализа.


# Что сдавать

В зависимости от вашего формата, можно сдавать:
- `.ipynb` с пояснениями и визуализацией;
- ссылку на Colab;
- файл `app.py` для Streamlit;
- HTML-файл из PyVis;
- PDF / презентацию, если вы делаете нарративную визуальную историю.

